
# StereoQueerEval — Phát hiện Stereotype & Hate Speech đa ngôn ngữ (Multi-task, from-scratch)

Phiên bản **adapt** của notebook `model-toxic.ipynb` (Jigsaw Toxic Comment Challenge)
sang bài toán **StereoQueerEval (SemEval 2027)**: phát hiện định kiến và hate speech
trong bình luận YouTube về chủ đề LGBTQIA+, gồm 3 ngôn ngữ **EN / IT / NL**.

### 3 subtask dùng chung 1 encoder
| Subtask | Nhãn | Loại | Loss |
|---|---|---|---|
| `stereotype` | `yes` / `no` | 1 nhãn binary | BCEWithLogits |
| `hate_speech` | `no` / `yes_implicit` / `yes_explicit` | 3 lớp | CrossEntropy |
| `target` | `none` hoặc `scope + identities` (l,g,b,t,q,i,a,nb,lgbtqia+) | 10 nhãn binary | BCEWithLogits |

### Những gì phải sửa so với notebook cũ (quan trọng!)
1. **3 đầu ra** thay vì 1 head 6 nhãn.
2. **Đầu vào** gồm `comment + title + description` (context giúp hiểu target của comment).
3. **Không xoá ký tự có dấu / đặc biệt** — `simple_clean` cũ dùng `[^a-zA-Z\s]`
   sẽ phá hỏng tiếng Ý (`perché`, `è`) và Hà Lan.
4. **Chia dữ liệu theo video** (`yt_title`) để tránh rò rỉ — vì title/description là INPUT.
5. **Không có test set** (ra mắt 01/2027) → đánh giá trên val split.
6. **1 vocab chung** cho cả 3 ngôn ngữ (1 model duy nhất).

---



## Dữ liệu
- 3 file TSV UTF-8: `StereoQueerEval_{EN,IT,NL}_training.tsv`.
- **Cảnh báo:** file có ký tự xuống dòng `\n` **bên trong field** → bắt buộc dùng
  `pandas.read_csv(sep='\t')`, không đọc bằng công cụ theo dòng.

| File | Số comment |
|---|---|
| EN | 2.989 |
| IT | 2.400 |
| NL | 2.238 |
---


In [ ]:

import os
import re
import glob
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Notebook nằm trong /notebook, dữ liệu nằm ở ../LGBT
candidates = [
    '../LGBT/*_training.tsv',
    'LGBT/*_training.tsv',
    '../**/StereoQueerEval_*_training.tsv',
]
paths = sorted({p for pat in candidates for p in glob.glob(pat, recursive=True)})
assert paths, 'Không tìm thấy file dữ liệu StereoQueerEval (*_training.tsv)'
print(paths)



## Chọn chế độ embedding & model
- **`EMBED_SOURCE = 'mmbert'`** (khuyến nghị) — dùng **mmBERT** (`jhu-clsp/mmbert-base`, gốc
  **ModernBERT 22 tầng**) **đông cứng / bán đông cứng**: mã hoá `comment [SEP] title [SEP] description`
  thành feature 768-d (max 512 token), rồi train Transformer head + 3 head (xem tuỳ chọn 2-pha bên dưới).
  Cần `pip install -U transformers`; lần đầu tải ~1.1GB model.
- **`EMBED_SOURCE = 'scratch'`** — học `nn.Embedding` từ đầu (giống notebook gốc); chọn RNN/RNN-LSTM/Transformer.


In [ ]:

EMBED_SOURCE = 'mmbert'   # 'mmbert' | 'scratch'

# --- Chỉ dùng khi EMBED_SOURCE == 'scratch' ---
train_on_RNNVanilla  = False
train_on_RNNLSTM     = False
train_on_Transformer = True

# --- Chỉ dùng khi EMBED_SOURCE == 'mmbert' ---
# True : mmBERT (đông cứng) NẰM TRONG model, dùng feature từng token -> TransformerEncoder nhỏ -> 3 head
# False: encode sẵn 1 lần -> mean-pool 768-d -> FeatureClassifier (MLP nhanh, ít RAM)
MMBERT_TRANSFORMER_HEAD = True

MMBERT_MODEL_NAME = 'jhu-clsp/mmbert-base'   # ModernBERT (22 tầng encoder), 768-d, context 8192
MMBERT_DIM = 768

# --- Fine-tune nhẹ phần đuôi mmBERT (chỉ áp dụng khi MMBERT_TRANSFORMER_HEAD) ---
MMBERT_UNFREEZE_LAYERS = 2   # mở N tầng ENCODER cuối của mmBERT để fine-tune; 0 = đông cứng hẳn
MMBERT_UNFREEZE_LR = 2e-5    # LR riêng cho phần mmBERT mở — nhỏ, tránh "phá" pretrained

# --- Huấn luyện 2 pha (chỉ khi MMBERT_TRANSFORMER_HEAD) ---
# True : pha 1 mmBERT FROZEN hẳn (để encoder + 3 head ổn định) -> pha 2 mở N tầng cuối fine-tune
# False: huấn luyện 1 pha duy nhất theo đúng MMBERT_UNFREEZE_LAYERS
TWO_PHASE = True
FREEZE_PHASE_EPOCHS = 20    # pha 1 (frozen): số epoch tối đa (early stopping vẫn áp dụng)
UNFREEZE_PHASE_EPOCHS = 20  # pha 2 (mở N tầng cuối): số epoch tối đa, LR head giảm còn 1e-5

FEAT_DIM = MMBERT_DIM        # kích thước feature encoder đông cứng

# Chế độ "feature trong model" (đúng) — dùng chung cho train/eval/predict
MM_TRANSFORMER = (EMBED_SOURCE == 'mmbert' and MMBERT_TRANSFORMER_HEAD)

assert EMBED_SOURCE in ('mmbert', 'scratch'), 'EMBED_SOURCE phải là "mmbert" hoặc "scratch"'



## 3 subtask: `st` / `hs` / `tg` — mỗi cột đo gì?
| Tên | Câu hỏi | Nhãn | Cách chấm |
|---|---|---|---|
| **`st`** (stereotype) | Câu chứa **định kiến** về người LGBTQIA+ không? | `yes`/`no` (1 bit) | accuracy + macro-F1 |
| **`hs`** (hate_speech) | Câu mang **ngôn từ thù ghét**, trực tiếp hay gián tiếp? | `no`/`yes_implicit`/`yes_explicit` (3 lớp) | accuracy + macro-F1 |
| **`tg`** (target) | Nạn nhân là **nhóm** hay **cá nhân**, thuộc nhóm định danh nào? | `none` hoặc `scope + identities` (`l,g,b,t,q,i,a,nb,lgbtqia+`) (9 bit + 1 scope) | exact-match chuỗi |

**Loss tổng = `1.5×st + 1.0×hs + 1.5×tg`.** Vì sao có trọng số? `hs` là CrossEntropy (3 lớp) cho
loss lớn hơn hẳn 2 đầu BCE (`st`, `tg`) — nếu cộng đều thì `hs` "nuốt" cả hai và `st`/`tg` gần như
không học. Boost `st` & `tg` lên 1.5 để 3 subtask tiến đều nhau.


In [ ]:

# 3 loss tương ứng 3 subtask
loss_st = nn.BCEWithLogitsLoss()   # stereotype: 1 nhãn
loss_hs = nn.CrossEntropyLoss()    # hate_speech: 3 lớp
loss_tg = nn.BCEWithLogitsLoss()   # target: 10 nhãn binary

# Trọng số khi cộng tổng loss. CE (hate_speech) trội hơn 2 BCE nên cần boost st & tg.
LOSS_ST_W, LOSS_HS_W, LOSS_TG_W = 1.5, 1.0, 1.5


def total_loss(st_logits, hs_logits, tg_logits, st, hs, tg):
    """1.5*st (BCE) + 1.0*hs (CE) + 1.5*tg (BCE)."""
    return (LOSS_ST_W * loss_st(st_logits.squeeze(-1), st)
            + LOSS_HS_W * loss_hs(hs_logits, hs)
            + LOSS_TG_W * loss_tg(tg_logits, tg))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang huấn luyện trên: {device}")


In [ ]:

dfs = []
for p in paths:
    lang = re.search(r'_([A-Z]{2})_training\.tsv$', p).group(1)
    df = pd.read_csv(p, sep='\t', quoting=1)   # quan trọng: field có xuống dòng
    df['lang'] = lang
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Tổng số comment: {len(df_all)}")
print(df_all.groupby('lang').size())


In [ ]:

def safe_clean(text):
    text = str(text).lower()
    # Chỉ gộp xuống dòng, KHÔNG xoá ký tự đặc biệt/có dấu (quan trọng với IT & NL)
    text = re.sub(r'[\n\t\r]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_all['text'] = (
    df_all['yt_comment'] + ' [SEP] ' + df_all['yt_title'] + ' [SEP] ' + df_all['yt_description']
).map(safe_clean)

print(df_all[['lang', 'text']].head(3).to_string())


In [ ]:

ID_ORDER = ['l', 'g', 'b', 't', 'q', 'i', 'a', 'nb', 'lgbtqia+']
SCOPE_DIM  = len(ID_ORDER)   # vị trí nhãn scope trong vector target
TARGET_DIM = SCOPE_DIM + 1   # 9 identity + 1 scope

HATE2IDX = {'no': 0, 'yes_implicit': 1, 'yes_explicit': 2}
IDX2HATE = {v: k for k, v in HATE2IDX.items()}


def encode_target(t):
    """'none' -> toàn 0; 'group_l,g' -> identity bits + scope."""
    v = np.zeros(TARGET_DIM, dtype=np.float32)
    if t == 'none':
        return v
    scope, ids = t.split('_', 1)
    v[SCOPE_DIM] = 1.0 if scope == 'group' else 0.0
    for i in ids.split(','):
        v[ID_ORDER.index(i)] = 1.0
    return v


def decode_target(v, thresh=0.5):
    """Ngược lại: vector -> string chuẩn theo thứ tự cố định; 'none' nếu rỗng."""
    ids = [ID_ORDER[i] for i in range(SCOPE_DIM) if v[i] >= thresh]
    if not ids:
        return 'none'
    scope = 'group' if v[SCOPE_DIM] >= thresh else 'individual'
    return f'{scope}_' + ','.join(ids)


df_all['st_y'] = (df_all['stereotype'] == 'yes').astype(np.float32).values
df_all['hs_y'] = df_all['hate_speech'].map(HATE2IDX).values
df_all['tg_y'] = [encode_target(t) for t in df_all['target']]

print(pd.crosstab(df_all['lang'], df_all['stereotype']))
print(pd.crosstab(df_all['lang'], df_all['hate_speech']))

# Kiểm tra round-trip encode/decode
v = encode_target('group_l,g')
print('group_l,g ->', v.tolist(), '->', decode_target(v))



## Chia dữ liệu theo video — chống rò rỉ
1 video (1 `yt_title`) chỉ xuất hiện ở 1 tập, vì title/description là **input**:
nếu cùng video nằm ở cả train lẫn val, model sẽ "nhìn thấy" context trước.


In [ ]:

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, val_idx = next(gss.split(df_all, groups=df_all['yt_title']))
df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_val   = df_all.iloc[val_idx].reset_index(drop=True)

overlap = len(set(df_train['yt_title']) & set(df_val['yt_title']))
print(f"Train: {len(df_train)} | Val: {len(df_val)} | video trùng lặp: {overlap}")


In [ ]:

def build_vocab(texts, max_vocab_size=30000):
    word_counts = Counter()
    for text in texts:
        word_counts.update(text.split())
    common_words = word_counts.most_common(max_vocab_size - 2)
    word_to_idx = {'<PAD>': 0, '<UNK>': 1}
    for i, (word, _) in enumerate(common_words):
        word_to_idx[word] = i + 2
    return word_to_idx

# Một vocab CHUNG cho cả 3 ngôn ngữ
vocab = build_vocab(df_train['text'], max_vocab_size=30000)
print("Vocab size:", len(vocab))


In [ ]:

class StereoQueerDataset(Dataset):
    def __init__(self, texts, st_labels, hs_labels, tg_labels, word_to_idx, max_len=256):
        self.texts = texts
        self.st_labels = st_labels
        self.hs_labels = hs_labels
        self.tg_labels = tg_labels
        self.word_to_idx = word_to_idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = [self.word_to_idx.get(w, 1) for w in str(self.texts[idx]).split()]

        # Padding / Truncating (giống notebook cũ)
        if len(ids) < self.max_len:
            ids += [0] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]

        return (torch.tensor(ids, dtype=torch.long),
                torch.tensor(self.st_labels[idx], dtype=torch.float32),
                torch.tensor(self.hs_labels[idx], dtype=torch.long),
                torch.tensor(self.tg_labels[idx], dtype=torch.float32))


In [ ]:

BATCH_SIZE = 32
MAX_LEN = 256

train_dataset = StereoQueerDataset(df_train['text'].values, df_train['st_y'].values,
                                   df_train['hs_y'].values, df_train['tg_y'].values, vocab, MAX_LEN)
val_dataset   = StereoQueerDataset(df_val['text'].values, df_val['st_y'].values,
                                   df_val['hs_y'].values, df_val['tg_y'].values, vocab, MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)



## mmBERT (frozen / bán đông cứng) làm backbone trong model — `EMBED_SOURCE == 'mmbert'`
Không dựng vocab/tokenizer riêng: `AutoTokenizer` của mmBERT (`jhu-clsp/mmbert-base`, gốc
**ModernBERT — 22 tầng encoder, 768-d, context 8192, sliding-window 128**) tạo `input_ids` +
`attention_mask`, literal `[SEP]` được thay bằng separator thật để model thấy
ranh giới comment/title/description. Dataset tokenize **MỘT LẦN** lúc khởi tạo rồi **cache**
(`MMBertSeqDataset`) — không tốn công mã hoá lại mỗi epoch.
Hai cách dùng (tuỳ `MMBERT_TRANSFORMER_HEAD`):

- **`True` (in-model):** `MMBertTransformerModel` lấy `last_hidden_state` `(B, L, 768)` ->
  `TransformerEncoder` tự dựng (padding mask = `attention_mask == 0`) -> masked mean-pool -> 3 head.
  mmBERT có thể **đông cứng hẳn** (forward trong `torch.no_grad()`) hoặc **mở N tầng encoder cuối**
  để fine-tune với LR nhỏ (`MMBERT_UNFREEZE_LAYERS`, `MMBERT_UNFREEZE_LR`), kèm chế độ **2-pha**
  (frozen trước đã ổn định head, rồi mới mở). Việc mở tầng dựa vào **chỉ số block trong tên tham số**
  (`encoder.layers.X.*`) nên không phụ thuộc thứ tự/thuộc tính module của từng phiên bản transformers.
- **`False` (pre-encode):** encode 1 lần -> mean-pool 768-d -> `FeatureClassifier` (MLP nhanh, ít RAM).


In [ ]:

class FeatureDataset(Dataset):
    """Dataset nhận feature vector có sẵn (không tokenize, không vocab)."""
    def __init__(self, feats, st_labels, hs_labels, tg_labels):
        self.feats = np.asarray(feats, dtype=np.float32)
        self.st_labels = np.asarray(st_labels)
        self.hs_labels = np.asarray(hs_labels)
        self.tg_labels = np.asarray(tg_labels)

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        return (torch.tensor(self.feats[idx]),
                torch.tensor(self.st_labels[idx], dtype=torch.float32),
                torch.tensor(self.hs_labels[idx], dtype=torch.long),
                torch.tensor(self.tg_labels[idx], dtype=torch.float32))


In [ ]:

class FeatureClassifier(nn.Module):
    """Encoder đông cứng (mmBERT) -> 3 head trên feature; chỉ phần này được train."""
    def __init__(self, in_dim=FEAT_DIM):
        super(FeatureClassifier, self).__init__()
        self.st_head = nn.Sequential(nn.Linear(in_dim, in_dim // 2), nn.ReLU(), nn.Dropout(0.3), nn.Linear(in_dim // 2, 1))
        self.hs_head = nn.Sequential(nn.Linear(in_dim, in_dim // 2), nn.ReLU(), nn.Dropout(0.3), nn.Linear(in_dim // 2, 3))
        self.tg_head = nn.Sequential(nn.Linear(in_dim, in_dim // 2), nn.ReLU(), nn.Dropout(0.3), nn.Linear(in_dim // 2, TARGET_DIM))

    def forward(self, x):
        return self.st_head(x), self.hs_head(x), self.tg_head(x)


In [ ]:

import re


def layer_index_in_name(name):
    """Lấy chỉ số tầng từ TÊN THAM SỐ (không cần biết cấu trúc module):
    - mmBERT/ModernBERT: 'layers.21.attn.Wqkv.weight'      -> 21
    - BERT cổ           : 'encoder.layer.9.attention...'   -> 9
    - không phải tầng   : 'embeddings.tok_embeddings'      -> None
    """
    m = re.search(r'(?:^|\.)(?:layers|layer|blocks)\.(\d+)\b', name)
    return int(m.group(1)) if m else None


def count_encoder_blocks(model):
    """Số tầng encoder, đếm theo chỉ số block trong TÊN THAM SỐ (không cần biết
    cấu trúc module của transformers — phiên bản nào cũng chạy)."""
    idx = {layer_index_in_name(n) for n, _ in model.named_parameters()}
    idx.discard(None)
    return len(idx)


def unfreeze_last_n(model, n):
    """Mở requires_grad cho n tầng encoder CUỐI theo chỉ số block trong tên tham số
    (ModernBERT: 'encoder.layers.X.*'; BERT cổ: 'encoder.layer.X.*'). Trả số tầng tìm thấy."""
    groups = {}
    for name, p in model.named_parameters():
        i = layer_index_in_name(name)
        if i is not None:
            groups.setdefault(i, []).append(p)
    if not groups:
        return 0
    for i in sorted(groups)[-n:]:
        for p in groups[i]:
            p.requires_grad = True
    return len(groups)


if EMBED_SOURCE == 'mmbert':
    from transformers import AutoTokenizer, AutoModel

    tok = AutoTokenizer.from_pretrained(MMBERT_MODEL_NAME)

    if MMBERT_TRANSFORMER_HEAD:
        # In-model: backbone giữ trên GPU suốt lúc train; freeze trước rồi mở N tầng cuối.
        mmbert_backbone = AutoModel.from_pretrained(MMBERT_MODEL_NAME).eval().to(device)
        for p in mmbert_backbone.parameters():
            p.requires_grad = False

        if MMBERT_UNFREEZE_LAYERS > 0 and not TWO_PHASE:
            n_blocks = unfreeze_last_n(mmbert_backbone, MMBERT_UNFREEZE_LAYERS)
            if n_blocks:
                print(f'mmBERT: {n_blocks} tầng -> fine-tune '
                      f'{min(MMBERT_UNFREEZE_LAYERS, n_blocks)} tầng cuối (LR {MMBERT_UNFREEZE_LR})')
            else:
                sample = ' | '.join(n for n, _ in list(mmbert_backbone.named_parameters())[:6])
                print('⚠️ Không tìn thấy tầng encoder trong tên tham số. Mẫu tên:', sample)
        elif TWO_PHASE:
            print('Phương án 2-pha: mmBERT đông cứng ở pha 1, sẽ mở '
                  f'{MMBERT_UNFREEZE_LAYERS} tầng cuối ở pha 2')
        else:
            print('mmBERT: đông cứng toàn bộ (feature extractor)')

        mmbert_train_feats = mmbert_val_feats = None
        print('MMBertTransformerModel: tokenize trong dataset')
    else:
        # Pre-encode 1 lần -> mean-pool 768-d cho FeatureClassifier (nhanh, ít RAM)
        mmbert_backbone = None
        mmbert = AutoModel.from_pretrained(MMBERT_MODEL_NAME).eval().to(device)

        def embed_mmbert(texts, batch_size=32, max_length=512):
            texts = [str(t).replace('[SEP]', tok.sep_token) for t in list(texts)]
            results = []
            for i in range(0, len(texts), batch_size):
                chunk = texts[i:i + batch_size]
                batch = tok(chunk, truncation=True, padding=True, max_length=max_length,
                            return_tensors='pt')
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.inference_mode():
                    hidden = mmbert(**batch).last_hidden_state.to(torch.float32)   # (B, L, 768)
                mask = batch['attention_mask'].unsqueeze(-1).to(torch.float32)
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
                results.append(pooled.cpu().numpy())
            return np.concatenate(results, axis=0).astype(np.float32)

        with torch.inference_mode():
            mmbert_train_feats = embed_mmbert(df_train['text'].values)
            mmbert_val_feats   = embed_mmbert(df_val['text'].values)
        del mmbert
        torch.cuda.empty_cache()
        print('mmBERT pooled:', mmbert_train_feats.shape, mmbert_val_feats.shape)
else:
    tok = mmbert_backbone = None
    mmbert_train_feats = mmbert_val_feats = None
    print('Bỏ qua mmBERT (EMBED_SOURCE == scratch)')



## MMBertSeqDataset + MMBertTransformerModel (`MMBERT_TRANSFORMER_HEAD = True`)
Dataset dùng `AutoTokenizer` của mmBERT để tokenize **toàn bộ 1 lần khi tạo dataset rồi cache**
`input_ids` + `attention_mask` (độ dài cố định theo `MAX_LEN`, `padding='max_length'`) —
mỗi epoch không còn chạy tokenizer, chỉ lấy slice từ cache. Không cần collate tự chế.

Model: feature `last_hidden_state (B, L, 768)` -> `TransformerEncoder` tự dựng (padding mask bằng
`attention_mask == 0`) -> masked mean-pool -> 3 head. mmBERT có thể xử lý kiểu:
- `MMBERT_UNFREEZE_LAYERS = 0` — đông cứng hẳn: forward trong `torch.no_grad()`, không build graph.
- `MMBERT_UNFREEZE_LAYERS = N` — **fine-tune N tầng encoder cuối**: mở `requires_grad`, build graph
  bình thường; phần còn lại vẫn frozen (không grad). Optimizer chia 2 nhóm LR: `MMBERT_UNFREEZE_LR`
  (mặc định 2e-5) cho phần mmBERT mở, LR thường cho encoder + 3 heads.


In [ ]:

class MMBertSeqDataset(Dataset):
    """Tokenize bằng AutoTokenizer của mmBERT MỘT LẦN lúc khởi tạo rồi cache
    `input_ids` + `attention_mask` (bỏ qua `build_vocab`); mỗi epoch không chạy tokenizer nữa."""

    def __init__(self, texts, st_labels, hs_labels, tg_labels, tok, max_len=256):
        self.st_labels = np.asarray(st_labels)
        self.hs_labels = np.asarray(hs_labels)
        self.tg_labels = np.asarray(tg_labels)
        self.ids, self.mask = self._encode(list(texts), tok, max_len)

    @staticmethod
    def _encode(texts, tok, max_len):
        texts = [str(t).replace('[SEP]', tok.sep_token) for t in texts]
        enc = tok(texts, truncation=True, max_length=max_len,
                  padding='max_length', return_tensors='pt')
        return enc['input_ids'], enc['attention_mask']

    def __len__(self):
        return len(self.st_labels)

    def __getitem__(self, idx):
        return (self.ids[idx],
                self.mask[idx],
                torch.tensor(self.st_labels[idx], dtype=torch.float32),
                torch.tensor(self.hs_labels[idx], dtype=torch.long),
                torch.tensor(self.tg_labels[idx], dtype=torch.float32))


class MMBertTransformerModel(nn.Module):
    """mmBERT (đông cứng) + TransformerEncoder tự dựng trên (B, L, 768) + 3 head."""
    def __init__(self, mmbert_model, d_model=768, num_heads=8, num_layers=2,
                 dim_feedforward=1024, dropout=0.3):
        super(MMBertTransformerModel, self).__init__()
        self.mmbert = mmbert_model   # backbone (mở/đông cứng do requires_grad, đổi được giữa 2 pha)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3 head inline (không phụ thuộc thứ tự khai báo make_transformer_head)
        self.st_head = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.ReLU(),
                                     nn.Dropout(dropout), nn.Linear(d_model // 2, 1))
        self.hs_head = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.ReLU(),
                                     nn.Dropout(dropout), nn.Linear(d_model // 2, 3))
        self.tg_head = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.ReLU(),
                                     nn.Dropout(dropout), nn.Linear(d_model // 2, TARGET_DIM))

    def forward(self, input_ids, attention_mask):
        # 1. mmBERT: chỉ lấy feature từng token, KHÔNG pool.
        #    - đông cứng hẳn     -> no_grad (không build graph qua backbone)
        #    - mở N tầng cuối    -> build graph bình thường (chỉ các tầng mở có grad)
        # Tự kiểm tra requires_grad mỗi lần forward nên đổi giữa 2 pha không cần thêm gì.
        backbone_trainable = any(p.requires_grad for p in self.mmbert.parameters())
        if backbone_trainable:
            seq_hidden = self.mmbert(input_ids=input_ids,
                                     attention_mask=attention_mask).last_hidden_state
        else:
            with torch.no_grad():
                seq_hidden = self.mmbert(input_ids=input_ids,
                                         attention_mask=attention_mask).last_hidden_state
        seq_hidden = seq_hidden.to(torch.float32)          # (B, L, 768)

        # 2. Transformer custom với padding mask (True = vị trí PAD)
        x = self.transformer_encoder(seq_hidden,
                                     src_key_padding_mask=(attention_mask == 0))

        # 3. Masked mean pooling (không cộng pad vào)
        m = attention_mask.unsqueeze(-1).float()
        x = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)

        return self.st_head(x), self.hs_head(x), self.tg_head(x)


In [ ]:

def make_head(hidden_dim, out_dim):
    """Khối classifier chuẩn của notebook cũ, dùng lại y hệt."""
    return nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(hidden_dim, hidden_dim // 2),
        nn.LayerNorm(hidden_dim // 2),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(hidden_dim // 2, out_dim),
    )


class VanillaRNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, device='cuda'):
        super(VanillaRNNModel, self).__init__()
        self.device = device
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.input_to_hidden  = nn.Linear(embedding_dim, hidden_dim)
        self.hidden_to_hidden = nn.Linear(hidden_dim, hidden_dim)

        # 3 đầu ra thay cho 1 head 6 nhãn
        self.st_head = make_head(hidden_dim, 1)
        self.hs_head = make_head(hidden_dim, 3)
        self.tg_head = make_head(hidden_dim, TARGET_DIM)

    def forward(self, x):
        batch_size, seq_len = x.size(0), x.size(1)
        h_t = torch.zeros(batch_size, self.hidden_dim, device=self.device)
        embedded = self.embedding(x)

        for t in range(seq_len):
            x_t = embedded[:, t, :]
            h_t = torch.tanh(self.input_to_hidden(x_t) + self.hidden_to_hidden(h_t))

        return self.st_head(h_t), self.hs_head(h_t), self.tg_head(h_t)



## Model RNN-LSTM (Bi-LSTM, giống notebook cũ)


In [ ]:

class PytorchRNNLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, device='cuda'):
        super(PytorchRNNLSTM, self).__init__()
        self.device = device
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(input_size=embedding_dim,
                            hidden_size=hidden_dim,
                            batch_first=True,
                            bidirectional=True)
        lstm_output_dim = hidden_dim * 2

        self.st_head = make_head(lstm_output_dim, 1)
        self.hs_head = make_head(lstm_output_dim, 3)
        self.tg_head = make_head(lstm_output_dim, TARGET_DIM)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        final_hidden_state = lstm_out[:, -1, :]  # bước thời gian cuối cùng

        return (self.st_head(final_hidden_state),
                self.hs_head(final_hidden_state),
                self.tg_head(final_hidden_state))



## Model Transformer (giống notebook cũ: Positional Encoding + TransformerEncoder + pooling)


In [ ]:

import math


class PytorchPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PytorchPositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


def make_transformer_head(in_dim, out_dim):
    return nn.Sequential(
        nn.Linear(in_dim, in_dim // 2),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_dim // 2, out_dim),
    )


class PytorchTransformerModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads=8, num_layers=4,
                 dim_feedforward=1024, dropout=0.3):
        super(PytorchTransformerModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoding = PytorchPositionalEncoding(embedding_dim, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.st_head = make_transformer_head(embedding_dim, 1)
        self.hs_head = make_transformer_head(embedding_dim, 3)
        self.tg_head = make_transformer_head(embedding_dim, TARGET_DIM)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)  # Global Average Pooling

        return self.st_head(x), self.hs_head(x), self.tg_head(x)



## Hàm đánh giá
- `stereotype`: accuracy + macro-F1
- `hate_speech`: accuracy + macro-F1 (3 lớp)
- `target`: **exact-match** giữa chuỗi dự đoán và chuỗi gốc (khớp cách chấm điểm SemEval)


In [ ]:

from sklearn.metrics import accuracy_score, f1_score


@torch.no_grad()
def predict(model, loader):
    model.eval()
    st_list, hs_list, tg_list = [], [], []
    for batch in loader:
        if MM_TRANSFORMER:
            ids, mask, _, _, _ = batch
            st_logits, hs_logits, tg_logits = model(ids.to(device), mask.to(device))
        else:
            texts, _, _, _ = batch
            st_logits, hs_logits, tg_logits = model(texts.to(device))
        st_list.append(torch.sigmoid(st_logits).cpu().numpy())
        hs_list.append(torch.argmax(hs_logits, dim=1).cpu().numpy())
        tg_list.append(torch.sigmoid(tg_logits).cpu().numpy())
    return (np.concatenate(st_list).ravel(),
            np.concatenate(hs_list),
            np.concatenate(tg_list))


def evaluate(model, df, loader):
    st_prob, hs_pred, tg_vec = predict(model, loader)
    st_pred = (st_prob >= 0.5).astype(int)
    tg_pred_str = np.array([decode_target(v) for v in tg_vec])

    metrics = {
        'st_acc': accuracy_score(df['st_y'].values, st_pred),
        'st_f1':  f1_score(df['st_y'].values, st_pred, average='macro', zero_division=0),
        'hs_acc': accuracy_score(df['hs_y'].values, hs_pred),
        'hs_f1':  f1_score(df['hs_y'].values, hs_pred, average='macro', zero_division=0),
        'tg_acc': (tg_pred_str == df['target'].values).mean(),
    }
    return metrics, (st_pred, hs_pred, tg_pred_str)


def print_metrics(metrics, name='VALIDATION'):
    print(f'--- {name} ---')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')



## Huấn luyện — vòng lặp dùng chung (giữ nguyên cách Early Stopping + save best của notebook cũ)


In [ ]:

import time

def build_optimizer(model, lr):
    """2 nhóm LR: mmBERT mở (LR nhỏ) tách khỏi encoder+heads (LR bình thường)."""
    if MM_TRANSFORMER:
        mmbert_prms = [p for n, p in model.named_parameters()
                       if p.requires_grad and n.startswith('mmbert.')]
        rest_prms = [p for n, p in model.named_parameters()
                     if p.requires_grad and not n.startswith('mmbert.')]
        if mmbert_prms:
            return torch.optim.Adam([
                {'params': mmbert_prms, 'lr': MMBERT_UNFREEZE_LR},
                {'params': rest_prms, 'lr': lr},
            ])
    return torch.optim.Adam(model.parameters(), lr=lr)


def train_model(model, tag, num_epochs=60, patience=7, lr=0.0001):
    optimizer = build_optimizer(model, lr)
    best_val_loss = float('inf')
    counter = 0
    early_stop = False

    print("Starting the training process with Early Stopping...")
    for epoch in range(num_epochs):
        if early_stop:
            print("🛑 Early stopping triggered. Training finished!")
            break

        start_time = time.time()

        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            if MM_TRANSFORMER:
                ids, mask, st, hs, tg = batch
                ids, mask = ids.to(device), mask.to(device)
                st_logits, hs_logits, tg_logits = model(ids, mask)
            else:
                texts, st, hs, tg = batch
                texts = texts.to(device)
                st_logits, hs_logits, tg_logits = model(texts)
            st, hs, tg = st.to(device), hs.to(device), tg.to(device)
            loss = total_loss(st_logits, hs_logits, tg_logits, st, hs, tg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                if MM_TRANSFORMER:
                    ids, mask, st, hs, tg = batch
                    ids, mask = ids.to(device), mask.to(device)
                    st_logits, hs_logits, tg_logits = model(ids, mask)
                else:
                    texts, st, hs, tg = batch
                    texts = texts.to(device)
                    st_logits, hs_logits, tg_logits = model(texts)
                st, hs, tg = st.to(device), hs.to(device), tg.to(device)
                val_loss += total_loss(st_logits, hs_logits, tg_logits, st, hs, tg).item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)

        metrics, _ = evaluate(model, df_val, val_loader)
        print(f"Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} "
              f"| Time: {time.time()-start_time:.1f}s")
        print_metrics(metrics, f'epoch {epoch+1}')

        # --- KIỂM TRA PATIENCE (giống notebook cũ) ---
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), f'{tag}.pt')
            counter = 0
        else:
            counter += 1
            print(f"⚠️ No improvement. Patience counter: {counter}/{patience}")
            if counter >= patience:
                early_stop = True


In [ ]:

final_tag = None

if EMBED_SOURCE == 'mmbert':
    if MMBERT_TRANSFORMER_HEAD:
        # mmBERT nằm trong model; dataset tokenize tại chỗ -> 5-tuple mỗi mẫu
        train_loader = DataLoader(MMBertSeqDataset(df_train['text'].values,
                                                   df_train['st_y'].values,
                                                   df_train['hs_y'].values,
                                                   df_train['tg_y'].values, tok, MAX_LEN),
                                  batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(MMBertSeqDataset(df_val['text'].values,
                                                   df_val['st_y'].values,
                                                   df_val['hs_y'].values,
                                                   df_val['tg_y'].values, tok, MAX_LEN),
                                  batch_size=BATCH_SIZE, shuffle=False)
        final_tag = 'best_mmbert_tf'
        model = MMBertTransformerModel(mmbert_backbone, d_model=MMBERT_DIM).to(device)

        def freeze_mmbert(n_unfreeze):
            """Đông cứng toàn bộ mmBERT, sau đó mở n_unfreeze tầng encoder cuối
            (thao tác theo chỉ số block trong TÊN THAM SỐ — version-proof)."""
            for p in mmbert_backbone.parameters():
                p.requires_grad = False
            if n_unfreeze > 0:
                n_blocks = unfreeze_last_n(mmbert_backbone, n_unfreeze)
                if n_blocks:
                    opened = min(n_unfreeze, n_blocks)
                    n_trainable = sum(1 for p in mmbert_backbone.parameters() if p.requires_grad)
                    print(f'> mmBERT: {n_blocks} tầng -> mở {opened} tầng cuối '
                          f'({n_trainable} params trainable)')
                else:
                    sample = ' | '.join(n for n, _ in list(mmbert_backbone.named_parameters())[:6])
                    print('⚠️ Không tìm thấy tầng encoder trong tên tham số. Mẫu tên:', sample)
            else:
                print('> mmBERT đông cứng toàn bộ')

        if TWO_PHASE:
            # PHA 1: mmBERT FROZEN hẳn, để encoder + 3 head ổn định trước
            freeze_mmbert(0)
            print(f'--- PHA 1/2 (frozen): tối đa {FREEZE_PHASE_EPOCHS} epoch, lr head 1e-4 ---')
            train_model(model, tag=final_tag, num_epochs=FREEZE_PHASE_EPOCHS, lr=0.0001)
            # PHA 2: nạp checkpoint tốt nhất pha 1, mở N tầng cuối, fine-tune LR nhỏ
            model.load_state_dict(torch.load(f'{final_tag}.pt', map_location=device))
            freeze_mmbert(MMBERT_UNFREEZE_LAYERS)
            print(f'--- PHA 2/2 (unfreeze {MMBERT_UNFREEZE_LAYERS} tầng cuối): tối đa '
                  f'{UNFREEZE_PHASE_EPOCHS} epoch, head lr 1e-5, mmBERT lr {MMBERT_UNFREEZE_LR} ---')
            final_tag = final_tag + '_ft'
            train_model(model, tag=final_tag, num_epochs=UNFREEZE_PHASE_EPOCHS, lr=0.00001)
        else:
            freeze_mmbert(MMBERT_UNFREEZE_LAYERS)
            train_model(model, tag=final_tag, lr=0.0001)
    else:
        # Mean-pool 768-d -> MLP 3 head (nhanh, ít RAM)
        train_loader = DataLoader(FeatureDataset(mmbert_train_feats, df_train['st_y'].values,
                                                 df_train['hs_y'].values, df_train['tg_y'].values),
                                  batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(FeatureDataset(mmbert_val_feats, df_val['st_y'].values,
                                                 df_val['hs_y'].values, df_val['tg_y'].values),
                                  batch_size=BATCH_SIZE, shuffle=False)
        final_tag = 'best_mmbert'
        model = FeatureClassifier(in_dim=FEAT_DIM).to(device)
        train_model(model, tag=final_tag, lr=0.001)

else:  # EMBED_SOURCE == 'scratch'
    if train_on_RNNVanilla:
        final_tag = 'best_rnn_vanilla'
        model = VanillaRNNModel(vocab_size=len(vocab), embedding_dim=128,
                                hidden_dim=256, device=device).to(device)
        train_model(model, tag=final_tag, lr=0.001)

    if train_on_RNNLSTM:
        final_tag = 'best_rnn_lstm'
        model = PytorchRNNLSTM(vocab_size=len(vocab), embedding_dim=128,
                               hidden_dim=256, device=device).to(device)
        train_model(model, tag=final_tag, lr=0.001)

    if train_on_Transformer:
        final_tag = 'best_transformer'
        model = PytorchTransformerModel(vocab_size=len(vocab), embedding_dim=256).to(device)
        train_model(model, tag=final_tag, lr=0.0001)

assert final_tag is not None, 'Chưa bật model nào (EMBED_SOURCE hoặc train_on_*)'
print(f"Đã huấn luyện xong -> {final_tag}.pt")


In [ ]:

model.load_state_dict(torch.load(f'{final_tag}.pt', map_location=device))

metrics, (st_pred, hs_pred, tg_pred_str) = evaluate(model, df_val, val_loader)
print_metrics(metrics, 'VALIDATION (overall)')

# Chi tiết từng ngôn ngữ
for lang in sorted(df_val['lang'].unique()):
    mask = df_val['lang'].values == lang
    sub = df_val[mask].reset_index(drop=True)
    if MM_TRANSFORMER:
        dset = MMBertSeqDataset(sub['text'].values, sub['st_y'].values,
                                sub['hs_y'].values, sub['tg_y'].values, tok, MAX_LEN)
        dload = DataLoader(dset, batch_size=BATCH_SIZE, shuffle=False)
    elif EMBED_SOURCE == 'mmbert':
        dset = FeatureDataset(mmbert_val_feats[mask], sub['st_y'].values,
                              sub['hs_y'].values, sub['tg_y'].values)
        dload = DataLoader(dset, batch_size=BATCH_SIZE, shuffle=False)
    else:
        dset = StereoQueerDataset(sub['text'].values, sub['st_y'].values,
                                  sub['hs_y'].values, sub['tg_y'].values, vocab, MAX_LEN)
        dload = DataLoader(dset, batch_size=BATCH_SIZE, shuffle=False)
    m, _ = evaluate(model, sub, dload)
    print(f'  [{lang}] ' + ' | '.join(f'{k}={v:.3f}' for k, v in m.items()))



## Xuất cấu hình & vocab (để app UI dùng lại sau này)


In [ ]:

import pickle

cfg = {
    'EMBED_SOURCE': EMBED_SOURCE,
    'MMBERT_TRANSFORMER_HEAD': MMBERT_TRANSFORMER_HEAD,
    'TWO_PHASE': TWO_PHASE,
    'FREEZE_PHASE_EPOCHS': FREEZE_PHASE_EPOCHS,
    'UNFREEZE_PHASE_EPOCHS': UNFREEZE_PHASE_EPOCHS,
    'MMBERT_UNFREEZE_LAYERS': MMBERT_UNFREEZE_LAYERS,
    'MMBERT_UNFREEZE_LR': MMBERT_UNFREEZE_LR,
    'ID_ORDER': ID_ORDER,
    'SCOPE_DIM': SCOPE_DIM,
    'TARGET_DIM': TARGET_DIM,
    'HATE2IDX': HATE2IDX,
    'MMBERT_MODEL_NAME': MMBERT_MODEL_NAME,
    'FEAT_DIM': FEAT_DIM,
    'MAX_LEN': MAX_LEN,
}
with open('stereoqueer_config.pkl', 'wb') as f:
    pickle.dump(cfg, f)
with open('stereoqueer_vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print('Đã lưu: stereoqueer_config.pkl + stereoqueer_vocab.pkl')



## Dự đoán mẫu & lưu kết quả (thay cho submission.csv — chưa có test set)


In [ ]:

results = df_val[['StereoQueerEval_id', 'lang', 'yt_comment', 'stereotype', 'hate_speech', 'target']].copy()
results['st_pred'] = st_pred
results['hs_pred'] = [IDX2HATE[int(i)] for i in hs_pred]
results['target_pred'] = tg_pred_str
results.to_csv('stereoqueer_val_predictions.csv', index=False)

pd.set_option('display.max_colwidth', 80)
print(results.head(10).to_string())
